1. Shape of embedding [n_t]
2. Shape of output[n_vocab]

Build the network: x_n -> f(x_n) -> error -> x_n-1 -> 
For each example, run the inference pass, then run the backward learning pass 

For inference: 
1. Get the input
2. Initialise the layers to a random value 
3. Pad the weights & the layers [do this beforehand]
4. Calculate the errors, pre-activation for all layers
5. Get the gradient
6. Update layers using gradient

For training: 
1. Get the errors & pre-activations for all layers 
2. Get the gradient
3. Update the weights using the gradient 

For per minibatch update training: 
1. Initialise the weights randomly
2. Get the training samples & fix input layers to that
3. Fix the input layer to that 
4. Add necessary padding
5. Run the traing cycle per sample
6. Batch the training update process across multiple samples using vmap

For each epochs - run for n epochs: 
1. Get the list of minibatches from the training batch
2. Run the training for each minibatch
3. Updates weights 

TODO (in order): 
1. Replicate results from the CFAIR training described in https://arxiv.org/pdf/2506.06332
2. Training details - how to interleave inference & weight updates, accelarating training via parallel kernels, batching over samples
3. Training over the minibatch - gradient is on the basis of avg. grad across the batch (but is avg the only way)
4. Can PC learning be applied to transformers -> learning attention is the issue, "attention-like patterns", where the entire text is processed via linear layers
5. Building an RNN network trained for auto-regressive text generation using PCN

In [11]:
import jax.numpy as jnp
import jax

class PCN: 
    num_layers = 0
    ## These are stored as list of jnp arrays, 
    # however will be padded & converted to jnp arrays 
    layer_sizes = (0.0,0.0,0.0)
    weights = []
    eta_inference = 0.1
    eta_learn = 0.1
    t_infer = 50

    def __init__(self,layer_sizes,weights): 
        self.layer_sizes = layer_sizes
        self.weights = weights
        self.num_layers = len(layers)

## Use a data-base of text; just what comes after in the alphabet 
## Use rev mode grad diff & update on that basis.

##Pads the layer & weights tensors given 
# the sizes may differ across layers: 
# Input is t which is a list of jnp Arrays of shape (nd_i) 
# or shape (nd_i,nd_i-1)
def build_padded_tensor(t,max_size,dim): 
    return []

# layers shape: [nl,nd_i]
# weights shape: [n_l-1,nd_i-1,nd_i]
## layers proceed from x_l.... x_0
## layers proceed from W_l-1 .... W_0
## Errors are from e_l-1....e_0
@jax.jit
def errorsnpreactivations(layers: jnp.array, weights: jnp.array):
    # indices = [range(5)]
    first_layer_index = layers.shape[1] - 1
    upper_layers = layers[0:first_layer_index-1,:]
    # upper layers shape: [n_l-1,nd]
    # weights shape: [n_l-1,nd,nd]
    # preactivation shape: [n_l-1,nd]
    ##TO FIX: Is the matrix multiplication as expected?
    preactivation = weights @ upper_layers
    activation = jax.nn.relu(preactivation)
    lower_layers = layers[1:first_layer_index,:]
    # errors shape: [n_l-1,nd]
    errors = activation - lower_layers
    return errors, preactivation

@jax.jit
def mse(errors: jnp.array): 
    jnp.sum(jnp.square(errors))

##Layers shape: [nl,nd]; without padding [nl,nd_i]
## Errors shape: [nl-1,nd]; without padding [nl-1,nd_i-1]
## Preactivation shape: [nl-1,nd]; without padding [nl-1,nd_i-1]
## Weights shape: [nl-1, nd, nd]; without padding [n_l-1,nd_i-1,nd_i]
@jax.jit
def update_layers(layers: jnp.array, weights: jnp.array, errors: jnp.array, preactivations: jnp.array, eta_inference):
    errors_shifted = jnp.pad(errors[0:len(errors)-1,:],((0,1),(0,0)),mode='empty')
    print("Errors shifted in update_layers",errors_shifted)
    first_layer_index = layers.shape[1] - 1
    upper_layers = layers[0:first_layer_index-1,:]
    drelu = jax.grad(jax.nn.relu)
    ##TO FIX: Potential bug here, the vector may be transposed along the batch dimension, avoid that:
    ##TO FIX: Is the matrix multiplication as expected?
    grad = errors_shifted - jnp.matrix_transpose(weights) @ (errors * drelu(preactivations))
    upper_layers = upper_layers - eta_inference * grad
    # Once done return the layers: 
    return jnp.append(upper_layers,[layers[len(layers)-1]])

##TODO: Just return the gradients here: 
@jax.jit
def update_weights(layers,weights,errors,preactivations,eta_learn):
    drelu = jax.grad(jax.nn.relu)
    first_layer_index = layers.shape[1] - 1
    upper_layers = layers[0:first_layer_index-1,:]
    ##TO FIX: Potential bug here, the vector may be transposed along the batch dimension, avoid that:
    ##TO FIX: Is the matrix multiplication as expected?
    grad = -(errors * drelu(preactivations)) @ jnp.transpose(upper_layers)
    return weights + (eta_learn * grad)

@jax.jit
def inference_updates(num_t,layers,weights,eta_inference):
    def inference_update(params,t):
        layers = params[0]
        weights = params[1]
        eta_inference = params[2]
        errors, preactivation = errorsnpreactivations(layers,weights)
        layers = update_layers(layers,weights,errors,preactivation,eta_inference)
        return ((layers, weights),)
    jax.lax.scan(inference_update,(layers,weights,eta_inference),range(num_t))

##TODO: just return the grads here? this might need re-structuring for batch updates: 
@jax.jit
def training_update(layers,weights,eta_train):
    errors, preactivation = errorsnpreactivations(layers,weights)
    ## Note: no need for updating weights here? 
    weights = update_weights(layers,weights,errors,preactivation,eta_train)
    return weights

@jax.jit
def train_sample(num_t,layers,weights,etas): 
    layers,weights = update_layers(num_t,layers,weights,etas[0])[0]
    weights = training_update(layers,weights,etas[1])

def initialise(): 
    key = jax.random.key(434)
    num_batch_size = 10
    num_input_size = 3072
    num_output_size = 10
    num_hidden_size = 100
    num_hidden_layers = 2
    pad_size = num_input_size
    input_layer = jnp.pad(jax.random.uniform(key,(num_batch_size,1,num_input_size)),((0,0),(0,0),(0,pad_size - num_input_size)),mode='empty')
    hidden_layers = jnp.pad(jax.random.uniform(key,(num_batch_size,num_hidden_layers,num_hidden_size)),((0,0),(0,0),(0,pad_size - num_hidden_size)),mode='empty')
    output_layer = jnp.pad(jax.random.uniform(key,(num_batch_size,1,num_output_size)),((0,0),(0,0),(0,pad_size - num_output_size)),mode='empty')
    layers = jnp.append(jnp.append(output_layer,hidden_layers,1),input_layer,1)
    input_weight_m = jnp.pad(jax.random.uniform(key,(1,num_hidden_size,num_input_size)),((0,0),(0,pad_size - num_hidden_size),(0,pad_size - num_input_size)),mode='empty')
    hidden_weight_m = jnp.pad(jax.random.uniform(key,(1,num_hidden_size,num_hidden_size)),((0,0),(0,pad_size - num_hidden_size),(0,pad_size - num_hidden_size)),mode='empty')
    output_weight_m = jnp.pad(jax.random.uniform(key,(1,num_output_size,num_hidden_size)),((0,0),(0,pad_size - num_output_size),(0,pad_size - num_hidden_size)),mode='empty')
    weights = jnp.append(jnp.append(output_weight_m,hidden_weight_m,0),input_weight_m,0)

    print("Layers: ",layers.shape)
    print("Weights: ",weights.shape)
    layers,weights
    
initialise()

Layers:  (10, 4, 3072)
Weights:  (3, 3072, 3072)
